**Table of contents**<a id='toc0_'></a>    
- 1. [Setup](#toc1_)    
  - 1.1. [Load Dataset](#toc1_1_)    
  - 1.2. [Configuration](#toc1_2_)    
- 2. [Load kết quả:](#toc2_)    
- 3. [Đánh giá](#toc3_)    

<!-- vscode-jupyter-toc-config
	numbering=true
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# 1. <a id='toc1_'></a>[Setup](#toc0_)

In [3]:
import torch

print(torch.__version__)
print(torch.version.cuda)

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

!nvidia-smi

2.10.0+cpu
None
/bin/bash: line 1: nvidia-smi: command not found


In [12]:
# Kiểm tra root_dir trên Kaggle
import os
print(os.listdir("/kaggle/input/datasets/nostagiguideus17"))


# Thiết lập để import source code
import sys
sys.path.append("/kaggle/input/datasets/nostagiguideus17/abstractive-summary-vers-transformer")


# Cài đặt thêm lib cần thiết
!pip install -q evaluate
!pip install -q bert_score

input_path = "/kaggle/input/datasets/nostagiguideus17/abstractive-summary-vers-transformer"

['abstractive-summary-vers-transformer']
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 742.8 kB/s eta 0:00:00a 0:00:01


## 1.1. <a id='toc1_1_'></a>[Load Dataset](#toc0_)

In [5]:
import pandas as pd
from datasets import Dataset

train_set_path = input_path + "/train-00000-of-00001.parquet"

dataset = Dataset.from_parquet(train_set_path)
df = dataset.to_pandas()
# df.to_csv('train.csv', index=False, encoding="utf-8-sig")

df.head()

Generating train split: 0 examples [00:00, ? examples/s]

,article,summary
0,Gần 20 sự kiện được tổ chức trên toàn thành ph...,Hà Nội tổ chức gần 20 sự kiện từ 19/4 đến 10/5...
1,"Được thành lập năm 1897 tại Đức, Kempinski Hot...",Kempinski Hotels là một thương hiệu nổi tiếng ...
2,"Ngoài di chuyển đến Tuần Châu bằng đường bộ, m...",Bài viết giới thiệu các hoạt động vui chơi giả...
3,"Với những tín đồ Phật giáo, bức tượng Phật ngọ...","Bức tượng Phật ngọc hòa bình thế giới, được vợ..."
4,Số liệu của Tổng cục Thống kê công bố sáng 29/...,"Trong tháng 4, Việt Nam tiếp tục đón khách quố..."


## 1.2. <a id='toc1_2_'></a>[Configuration](#toc0_)

In [6]:
from src.interfaces import ModelConfig
import torch

config = ModelConfig(
    lowercase = True, 
    dropout_prob = 0.1,

    vocab_size = 30000,
    embed_dim= 512,
    max_target_length = 1000,
    
    evaluation_config = {
            "rouge": {
                "rouge_types": ["rouge1", "rouge2", "rouge3", "rougeLsum"],
                "use_stemmer": True
            },
            "bertscore": {
                # Vì tóm tắt tiếng Việt, ta nên dùng PhoBERT để đo khoảng cách ngữ nghĩa
                "model_type": "vinai/phobert-base", 
                "num_layers": 4 # Tham số sâu của BERTScore (optional)
            }
        }
)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

# 2. <a id='toc2_'></a>[Load kết quả:](#toc0_)

Ở đây ta có thể sử dụng mô hình dự đoán trực tiếp, hoặc load kết quả từ file.

Đầu vào gồm:
    - predictions: [List, pd.Series] văn bản dự đoán (bản do mô hình viết)
    - references:  [List, pd.Series] văn bản đối chiếu (bản tóm tắt chuẩn của dataset)
    - original:    [List, pd.Series] văn bản gốc chưa tóm tắt -> Để nếu cần khi đánh giá bằng LLM (đang code)

In [1]:
predictions = [
    "Việt Nam đã xuất sắc giành chiến thắng trong trận đấu tối qua.",
    "Thời tiết hôm nay rất đẹp."
]
references = [
    "Đội tuyển Việt Nam giành chiến thắng thuyết phục vào tối hôm qua.",
    "Hôm nay trời nắng và thời tiết rất đẹp."
]

# 3. <a id='toc3_'></a>[Đánh giá](#toc0_)

In [ ]:
from src.evaluate import MultiEvaluator

evaluator = MultiEvaluator(config.evaluation_config)

In [ ]:
results = evaluator.score_batch(predictions, references)

results.to_csv("kaggle/working/results.csv")

[Info] Đang tải các mô hình đánh giá. Việc này có thể mất chút thời gian...
[Info] Đã tải xong metric: rouge
[Info] Đã tải xong metric: bertscore


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: vinai/phobert-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
lm_head.layer_norm.weight       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
lm_head.layer_norm.bias         | UNEXPECTED |  | 
lm_head.decoder.bias            | UNEXPECTED |  | 
lm_head.decoder.weight          | UNEXPECTED |  | 
lm_head.dense.bias              | UNEXPECTED |  | 
lm_head.dense.weight            | UNEXPECTED |  | 
lm_head.bias                    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,rouge1,rouge2,rouge3,rougeLsum,bertscore_precision,bertscore_recall,bertscore_f1
0,0.697674,0.390244,0.256410,0.55814,0.693120,0.689086,0.691097
1,0.800000,0.608696,0.380952,0.56000,0.693502,0.848612,0.763256


model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def visualize_rouge(df: pd.DataFrame, columns: list, fig_name:str = "Evaluation Scores") -> plt.Figure:
    """
    Vẽ biểu đồ cột thể hiện giá trị Min, Mean, Max của các độ đo ROUGE.
    
    Args:
        df: DataFrame chứa kết quả đánh giá.
        columns: Danh sách tên các cột cần vẽ (VD: ['rouge1', 'rouge2', 'rouge3', 'rougeLsum']).
        
    Returns:
        fig: Đối tượng Figure của matplotlib.
    """
    # 1. Trích xuất dữ liệu thống kê
    min_vals = df[columns].min()
    mean_vals = df[columns].mean()
    max_vals = df[columns].max()
    
    # 2. Thiết lập thông số tọa độ cho các cột
    x = np.arange(len(columns))  # Vị trí tâm của từng cụm (tick marks)
    width = 0.25                 # Độ rộng của mỗi cột
    
    # 3. Khởi tạo Figure và Axes
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # 4. Vẽ 3 nhóm cột (Min, Mean, Max)
    # Dịch trục x sang trái cho cột Min, giữ nguyên cho Mean, và dịch sang phải cho Max
    rects_min = ax.bar(x - width, min_vals, width, label='Min', color='#E63946')
    rects_mean = ax.bar(x, mean_vals, width, label='Mean', color='#2A9D8F')
    rects_max = ax.bar(x + width, max_vals, width, label='Max', color='#0077B6')
    
    # 5. Hàm phụ để ghi giá trị lên đỉnh cột
    def autolabel(rects):
        for rect in rects:
            height = rect.get_height()
            # Ghi text tại vị trí tâm của cột, nhích lên trên một chút (xytext=(0, 3))
            ax.annotate(f'{height:.3f}',
                        xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 3),  
                        textcoords="offset points",
                        ha='center', va='bottom', fontsize=9)
            
    # Áp dụng ghi chú cho cả 3 nhóm cột
    autolabel(rects_min)
    autolabel(rects_mean)
    autolabel(rects_max)
    
    # 6. Trang trí biểu đồ
    ax.set_ylabel('Scores')
    ax.set_title(fig_name)
    ax.set_xticks(x)
    ax.set_xticklabels(columns)
    ax.legend(loc='upper right')
    
    # Mở rộng giới hạn trục Y thêm một chút để không bị lẹm phần text trên đỉnh cột Max
    ax.set_ylim(0, 1.1)
    
    # Thêm lưới mờ theo trục ngang cho dễ nhìn
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    
    plt.tight_layout()
    
    return fig

In [ ]:
fig = visualize_rouge(results, columns = ['rouge1', 'rouge2', 'rouge3', 'rougeLsum', 'bertscore_f1'])